# CADHLLM Phase IV — LoRA-B 訓練（Colab）

> **Base Model**: `unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit`  
> **任務**: subsystems + 環境 → Assembly Plan JSON  
> **產出**: LoRA-B adapter → `outputs/cadhllm_lora_b/`

---

**使用方式**：依序執行每個 Cell。確認 Runtime 已選擇 **GPU（T4 或 A100）**。

## 1. 掛載 Google Drive

In [ ]:
import os
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

TRAINING_DIR = "/content/drive/MyDrive/CADHLLM/training"
assert Path(TRAINING_DIR).exists(), (
    f"[FAIL] {TRAINING_DIR} 不存在 — 請把本地 training/ 整個資料夾上傳到 Drive 此路徑"
)
os.chdir(TRAINING_DIR)
print(f"OK 工作目錄: {os.getcwd()}")

# 防呆:必要檔案是否齊全
REQUIRED_FILES = [
    "trainer.py", "trainer_b.py",
    "data_generator_b.py", "data_generator_b_helpers.py",
    "_registry_enclosure_relation.json",  # SSOT snapshot (Colab fallback)
    "config.py", "cadhllm_hparams.py",
    "data/cadhllm_lora_b_ch3.jsonl",  # v2 訓練資料 200 行
]
missing = [f for f in REQUIRED_FILES if not Path(f).exists()]
if missing:
    print(f"\n[FAIL] 缺少檔案: {missing}")
    print("請從本地 repo 重新上傳整個 training/ 資料夾覆蓋 Drive")
    raise FileNotFoundError(missing)
print("OK 所有必要檔案就位:")
for f in REQUIRED_FILES:
    size = Path(f).stat().st_size
    print(f"    {f} ({size:,} bytes)")


## 2. 安裝依賴

In [ ]:
!pip install -q unsloth transformers peft trl datasets accelerate bitsandbytes sentencepiece

## 3. 環境檢查

In [ ]:
from trainer_b import report_environment

info = report_environment()
print("=" * 50)
print(" 環境摘要")
print("=" * 50)
for k, v in info.items():
    print(f"  {k:22s}: {v}")
print("=" * 50)

assert info["cuda_available"], "❌ 請確認 Runtime → Change runtime type → GPU"
print(f"\n✅ GPU 就緒：{info['device_name']}（{info['vram_gb']} GB）")

## 4. 訓練超參

**注意**: DATA_SIZE 已移除 — Cell 10 直接讀 `data/cadhllm_lora_b_ch3.jsonl` (v2 200 行),
不再用 `gen.generate_synthetic_data()` 重生資料。

In [ ]:
MAX_STEPS = 200          # 訓練步數 (200 對 200 樣本 ~= 1 epoch; 提到 400 ~= 2 epoch)
LORA_R = 16              # LoRA rank (v2 schema 4 enum, r=16 應足夠)
LORA_ALPHA = 32          # LoRA alpha (= 2 x rank 常規)
LORA_DROPOUT = 0.05
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 2048    # 訓練序列長度上限
OUTPUT_DIR = "outputs/cadhllm_lora_b"


## 5. 載入訓練資料（v2 — 從 jsonl 讀檔繞過 silent fallback）

> **修正歷史**：原 cell-10 用 `gen.generate_synthetic_data(...)` 重生資料，但 Colab Drive 沒 `lib/registry.py` → helper 內 ImportError silent fallback → 591 elements 全 internal → LoRA-B 沒學到 v2 schema。  
> 2026-05-15 commit e13a2bf 改 fallback 為 explicit stderr WARNING；本 cell 改讀本地預生 jsonl，徹底繞過 Drive 環境問題。

In [ ]:
import json
from pathlib import Path
from collections import Counter

JSONL_PATH = Path(TRAINING_DIR) / "data" / "cadhllm_lora_b_ch3.jsonl"
assert JSONL_PATH.exists(), f"找不到 {JSONL_PATH};請先把本地 training/data/cadhllm_lora_b_ch3.jsonl 上傳到 Drive 此路徑"

dataset = []
with open(JSONL_PATH, encoding="utf-8") as f:
    for ln in f:
        ln = ln.strip()
        if ln:
            dataset.append(json.loads(ln))

print(f"OK 載入 {len(dataset)} 行 LoRA-B v2 訓練資料 from {JSONL_PATH}")
assert len(dataset) >= 100, f"預期至少 100 行,實際 {len(dataset)}"

first = dataset[0]
msgs = first["messages"]
asst_content = next(m["content"] for m in msgs if m["role"] == "assistant")
assert "enclosure_relation" in asst_content, "第一筆 assistant 不含 enclosure_relation 字串"
print("OK schema 驗證: assistant content 含 enclosure_relation 字串")

rel_cnt = Counter()
for s in dataset:
    user = next((m for m in s["messages"] if m["role"] == "user"), {})
    if "<|im_start|>plan" not in user.get("content", ""):
        continue
    asst = next((m for m in s["messages"] if m["role"] == "assistant"), None)
    if not asst:
        continue
    try:
        plan = json.loads(asst["content"])
    except json.JSONDecodeError:
        continue
    for el in plan.get("elements", []):
        rel_cnt[el.get("enclosure_relation", "MISSING")] += 1

print()
print(f"--- enclosure_relation 分布 ({sum(rel_cnt.values())} elements) ---")
for k, v in rel_cnt.most_common():
    print(f"  {k:12} {v}")
unique = set(rel_cnt) - {"MISSING"}
assert len(unique) >= 2, f"只有 {len(unique)} 個 enum 值,訓練無多樣性 - 本地重新生 jsonl"
print(f"OK enum 涵蓋率: {len(unique)}/5 = {sorted(unique)}")

print()
print("--- 第一筆 system message (前 200 字) ---")
print(msgs[0]["content"][:200])
print()
print("--- 第一筆 user prompt (前 300 字) ---")
print(msgs[1]["content"][:300])
print()
print("--- 第一筆 assistant completion (前 400 字,含 enclosure_relation) ---")
print(asst_content[:400])


## 6. 開始訓練

資料量少，預估 T4 約 20-40 分鐘，A100 約 5-10 分鐘。

In [ ]:
from pathlib import Path
from trainer_b import train

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

result = train(
    data=dataset,
    output_dir=OUTPUT_DIR,
    max_steps=MAX_STEPS,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    learning_rate=LEARNING_RATE,
)

print("\n" + "=" * 50)
print(" LoRA-B 訓練完成")
print("=" * 50)
print(f"  模式      : {result['mode']}")
print(f"  耗時      : {result['elapsed_s']/60:.1f} 分鐘")
print(f"  train/eval: {result.get('train_size','?')} / {result.get('eval_size','?')}")
print(f"  best_step : {result.get('best_step','?')}")
print(f"  best_loss : {result.get('best_eval_loss','?')}")
print(f"  輸出      : {result['output_dir']}")
print("=" * 50)


## 7. 快速推論測試

用訓練好的 LoRA-B adapter 跑一筆組裝決策推論：

In [ ]:
from unsloth import FastLanguageModel
import json

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=OUTPUT_DIR,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

# 3 範本 cross-validate (涵蓋不同元件組合,含 panel/external 元件)
TEST_TEMPLATES = [
    {
        "name": "auto_plant_waterer",
        "components": [
            ("Arduino-Uno-class", 25, 250),
            ("USB-5V-class", 15, 0),                # external
            ("Button-class", 1.5, 0),               # panel
            ("Sensor-SoilMoisture-class", 5, 0),
            ("Pump-Water-class", 45, 2000),
            ("Relay-Module-class", 18, 400),
        ],
        "env": "soil_contact", "ip": "IP20",
    },
    {
        "name": "smart_nightlight",
        "components": [
            ("ESP32-class", 10, 500),
            ("Lighting-LED-RGB-class", 8, 700),     # panel
            ("Sensor-Light-class", 3, 0),
            ("Switch-class", 5, 0),                 # panel
            ("USB-5V-class", 15, 0),                # external
        ],
        "env": "indoor_dry", "ip": "IP20",
    },
    {
        "name": "music_box",
        "components": [
            ("Arduino-Uno-class", 25, 250),
            ("Buzzer-Active-class", 5, 100),        # panel
            ("Button-class", 1.5, 0),               # panel
            ("Potentiometer-class", 5, 0),          # panel
            ("Battery-AA-class", 100, 0),           # external
        ],
        "env": "indoor_dry", "ip": "IP20",
    },
]

SYSTEM_MSG = (
    "你是 Phase IV Layer 2 階層式組裝決策師 (Plan 階段)。"
    "根據子系統列表、環境條件與物理約束,輸出 PlanJSON (高層決策)。"
    "決策因子:物理平衡、熱源管理、光照方向、結構強度、線路最短、維護性。"
    "每個元件需標註 enclosure_relation:internal=殼內 / breadboard=焊主板 / "
    "panel=穿殼開窗 / external=殼外 / embedded=結構體內。"
    "只輸出 JSON 物件,不要 Markdown。"
)
ENUM_SET = {"internal", "breadboard", "panel", "external", "embedded"}

results = []
for tpl in TEST_TEMPLATES:
    print(f"\n{'='*60}\n推理: {tpl['name']}\n{'='*60}")
    subs = ", ".join(f"{c}(weight={w}g, thermal={t}mW)" for c, w, t in tpl["components"])
    total_w = sum(w for _, w, _ in tpl["components"])
    total_t = sum(t for _, _, t in tpl["components"])
    msgs = [
        {"role": "system", "content": SYSTEM_MSG},
        {"role": "user", "content": (
            f"<|im_start|>plan\n專案:{tpl['name']}\n子系統:{subs}\n"
            f"總重量:{total_w}g,總發熱:{total_t}mW\n"
            f"環境:{tpl['env']},防水:False,IP:{tpl['ip']}\n"
            "外殼尺寸約束:compact (<=150mm)"
        )},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs, max_new_tokens=1500,
        temperature=0.5, min_p=0.1, do_sample=True,
    )
    resp = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    resp_clean = resp.split("<|eot_id|>")[0] if "<|eot_id|>" in resp else resp

    try:
        plan = json.loads(resp_clean)
    except json.JSONDecodeError as e:
        print(f"  [FAIL] JSON 解析失敗: {e}")
        print(f"  raw 前 500 字: {resp[:500]}")
        results.append({"tpl": tpl["name"], "ok": False, "reason": "json_parse_fail"})
        continue

    elements = plan.get("elements", [])
    rels = [el.get("enclosure_relation") for el in elements]
    n_missing = sum(1 for r in rels if r is None)
    n_invalid = sum(1 for r in rels if r is not None and r not in ENUM_SET)
    n_ok = sum(1 for r in rels if r in ENUM_SET)

    print(f"  elements: {len(elements)}, OK: {n_ok}, missing: {n_missing}, invalid: {n_invalid}")
    print(f"  enclosure_relation 值: {rels}")

    results.append({
        "tpl": tpl["name"],
        "ok": (n_missing == 0 and n_invalid == 0 and len(elements) > 0),
        "n_elements": len(elements), "n_ok": n_ok,
        "n_missing": n_missing, "n_invalid": n_invalid, "rels": rels,
    })

# 總結 + 嚴格 assertion
print(f"\n{'='*60}\n總結\n{'='*60}")
n_pass = sum(1 for r in results if r["ok"])
for r in results:
    flag = "OK" if r["ok"] else "FAIL"
    print(f"  [{flag}] {r['tpl']}: {r.get('n_ok','?')}/{r.get('n_elements','?')}")
print(f"\n{n_pass}/{len(results)} 範本通過 v2 schema 驗收")

if n_pass < len(results):
    print("\n[FAIL] LoRA-B 未完整學會 v2 schema — 不要下載 adapter,先 debug:")
    print("   1. 檢查 Cell 10 輸出:enum 涵蓋率 >= 4/5? 分布有 panel/external 嗎?")
    print("   2. 提高 MAX_STEPS 到 400 或 LORA_R 到 32 後重訓")
    print("   3. 確認 _registry_enclosure_relation.json 在 Drive 上且 mapping 非空")
    raise AssertionError(f"{len(results)-n_pass} 範本 FAIL")
print("\nOK 全部範本通過,可進 Cell 16 下載 adapter")

## 8. 下載 Adapter

執行後從瀏覽器下載 zip，解壓到主專案 `saved_model/cadhllm_lora_b/`（對齊 `adapter_manager._adapter_path` 讀取路徑；會覆蓋既有 B 案版本）。

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

out_path = Path(OUTPUT_DIR)
assert out_path.exists(), f"[FAIL] {OUTPUT_DIR} 不存在 — Cell 12 訓練未完成"
zip_path = shutil.make_archive(
    base_name="cadhllm_lora_b",
    format="zip",
    root_dir=out_path.parent,        # 'outputs/'
    base_dir=out_path.name,          # 'cadhllm_lora_b/'
)
size_mb = Path(zip_path).stat().st_size / 1024 / 1024
print(f"OK 已打包: {zip_path} ({size_mb:.1f} MB)")
print("下載後解壓到本地 saved_model/cadhllm_lora_b/ 覆蓋既有版")
files.download(zip_path)
